# 01 – Data

Turns the raw TidyTuesday BoardGameGeek files (`data/raw/`) into one analysis dataset
(`data/processed/games_clean.csv`) with one row per game and all variables used in notebooks 02 and 03.

1. Load and merge `details.csv` and `ratings.csv`
2. Clean impossible values
3. Select the sample (N per step is saved for the Methods section)
4. Build the variables
5. Sanity checks and save

In [114]:
import ast
import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "output" / "tables"
PROCESSED.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# Sample: games published 1950-2019 (reasons in section 3)
MIN_YEAR, MAX_YEAR = 1950, 2019
SNAPSHOT_YEAR = 2022  # the data were scraped in January 2022

# Expansion items listed under at least this many base games are generic promo packs
# (e.g. advent calendars), not expansions of one specific game
SHARED_ITEM_MIN_GAMES = 5

# H3 moderator, fixed on theoretical grounds before estimation: genres whose business model
# is built around a stream of add-ons from the start (scenario series, figure packs, boosters)
PLANNED_CATEGORIES = ["Wargame", "Miniatures", "Collectible Components"]
PLANNED_MECHANICS = ["Scenario / Mission / Campaign Game"]

# Placeholder "publishers" on BoardGameGeek that are not real firms
NON_PUBLISHERS = ["(Self-Published)", "(Web published)", "(Unknown)", "(Public Domain)"]

TOP_N = 15  # most common categories and mechanics turned into dummies (exploratory analysis)

pd.set_option("display.max_columns", 50)

## 1. Load and merge

In [115]:
details = pd.read_csv(RAW / "details.csv")
ratings = pd.read_csv(RAW / "ratings.csv")
assert details["id"].is_unique and ratings["id"].is_unique
print(f"details.csv: {len(details):,} games, ratings.csv: {len(ratings):,} games")
print(f"In ratings without details record (no expansion or ownership data): {(~ratings['id'].isin(details['id'])).sum()}")

games = details.merge(ratings[["id", "average", "users_rated"]], on="id", how="inner", validate="one_to_one")
games = games.rename(columns={
    "primary": "name",
    "boardgamecategory": "categories",
    "boardgamemechanic": "mechanics",
    "boardgameexpansion": "expansions",
    "boardgamepublisher": "publishers",
})
games = games[["id", "name", "yearpublished", "minplayers", "maxplayers", "playingtime", "minage",
               "categories", "mechanics", "expansions", "publishers", "owned", "average", "users_rated"]]
print(f"Merged: {len(games):,} games")

details.csv: 21,631 games, ratings.csv: 21,831 games
In ratings without details record (no expansion or ownership data): 200
Merged: 21,631 games


The list columns are stored as text, e.g. `"['Dice', 'Fantasy']"`. They are turned into Python lists;
a missing value means the game has no entry, so it becomes an empty list.

In [116]:
LIST_COLUMNS = ["categories", "mechanics", "expansions", "publishers"]
for col in LIST_COLUMNS:
    games[col] = games[col].map(lambda text: ast.literal_eval(text) if isinstance(text, str) else [])
games.head(3)

,id,name,yearpublished,minplayers,maxplayers,playingtime,minage,categories,mechanics,expansions,publishers,owned,average,users_rated
0,30549,Pandemic,2008,2,4,45,8,[Medical],"[Action Points, Cooperative Game, Hand Managem...",[Pandemic: Gen Con 2016 Promos – Z-Force Team ...,"[Z-Man Games, Albi, Asmodee, Asmodee Italia, A...",168364,7.59,108975
1,822,Carcassonne,2000,2,5,45,7,"[City Building, Medieval, Territory Building]","[Area Majority / Influence, Map Addition, Tile...","[20 Jahre Darmstadt Spielt, Apothecaries (fan ...","[Hans im Glück, 999 Games, Albi, Bard Centrum ...",161299,7.42,108738
2,13,Catan,1995,3,4,120,10,"[Economic, Negotiation]","[Dice Rolling, Hexagon Grid, Income, Modular B...","[20 Jahre Darmstadt Spielt, Brettspiel Advents...","[KOSMOS, 999 Games, Albi, Asmodee, Astrel Game...",167733,7.14,108024


## 2. Impossible values

On BoardGameGeek a value of 0 for player counts, playing time or minimum age means "unknown", so it is
set to missing. Extreme values (e.g. `maxplayers = 999`) are capped when the controls are built.

In [117]:
ZERO_IS_MISSING = ["minplayers", "maxplayers", "playingtime", "minage"]

print("Values equal to 0:")
print((games[ZERO_IS_MISSING] == 0).sum())

games[ZERO_IS_MISSING] = games[ZERO_IS_MISSING].replace(0, np.nan)
games[["yearpublished", "owned", "users_rated", "average"] + ZERO_IS_MISSING].describe().T

Values equal to 0:
minplayers       44
maxplayers      164
playingtime     774
minage         1301
dtype: int64


,count,mean,std,min,25%,50%,75%,max
yearpublished,21631.0,1986.094910,210.042496,-3500.00,2001.00,2011.00,2017.00,2023.00
owned,21631.0,1487.924553,5395.077773,0.00,150.00,322.00,903.50,168364.00
users_rated,21631.0,874.548518,3695.946026,30.00,57.00,124.00,397.00,108975.00
average,21631.0,6.417249,0.929345,1.04,5.83,6.45,7.04,9.57
minplayers,21587.0,2.011118,0.683668,1.00,2.00,2.00,2.00,10.00
maxplayers,21467.0,5.753109,15.151687,1.00,4.00,4.00,6.00,999.00
playingtime,20857.0,93.867958,544.370725,1.00,30.00,45.00,90.00,60000.00
minage,20330.0,10.226463,2.794918,1.00,8.00,10.00,12.00,25.00


## 3. Sample

`ratings.csv` only lists ranked games, which on BoardGameGeek requires at least 30 user ratings, so that
threshold needs no extra filter (checked below). Three selection steps follow:

1. **Unknown year.** A publication year of 0 means "unknown"; negative years are ancient games.
2. **Published before 1950.** The hypotheses assume that one publisher owns the base game and decides
   whether to release expansions after seeing how it sells. Games from before 1950 are mostly
   traditional or public-domain games (e.g. Chess, Go, Backgammon) and early mass-market classics that
   are sold by many publishers at once. For these games an "expansion" is not the decision of one firm,
   and the publisher controls are not meaningful (see the publisher counts below).
3. **Published after 2019.** Recent games have had little time to receive expansions.

In [118]:
assert (games["users_rated"] >= 30).all()

n_publishers = details.set_index("id")["boardgamepublisher"].map(
    lambda text: len(ast.literal_eval(text)) if isinstance(text, str) else 0)
year = games.set_index("id")["yearpublished"]
print("Median number of listed publishers per game:")
print(f"    published before {MIN_YEAR} (known year): {n_publishers[year[(year != 0) & (year < MIN_YEAR)].index].median():.0f}")
print(f"    published {MIN_YEAR}-{MAX_YEAR}: {n_publishers[year[year.between(MIN_YEAR, MAX_YEAR)].index].median():.0f}")

steps = [("Merged details + ratings (ranked games, at least 30 ratings)", games)]
games = games[games["yearpublished"] != 0]
steps.append(("Publication year known", games))
games = games[games["yearpublished"] >= MIN_YEAR]
steps.append((f"Published {MIN_YEAR} or later", games))
games = games[games["yearpublished"] <= MAX_YEAR].reset_index(drop=True)
steps.append((f"Published {MAX_YEAR} or earlier", games))

sample = pd.DataFrame({"step": [step for step, _ in steps], "n_games": [len(games) for _, games in steps]})
sample["n_dropped"] = (-sample["n_games"].diff()).fillna(0).astype(int)
sample.to_csv(TABLES / "sample_filters.csv", index=False)
sample

Median number of listed publishers per game:
    published before 1950 (known year): 8
    published 1950-2019: 1


,step,n_games,n_dropped
0,"Merged details + ratings (ranked games, at lea...",21631,0
1,Publication year known,21447,184
2,Published 1950 or later,21235,212
3,Published 2019 or earlier,19545,1690


## 4. Variables

**Outcome.** Installed base = number of users who own the game, used as `log(1 + owned)` because it is
strongly right-skewed. `log(1 + users_rated)` is an alternative outcome for the robustness check.

**Expansions.** BoardGameGeek also lists promos and generic promo packs that appear under many base
games. The main measure `n_expansions` leaves these out and only keeps *core* expansions. `n_expansions_all` counts
every listed item and is used as a robustness check.

In [119]:
games["log_owned"] = np.log1p(games["owned"])
games["log_users_rated"] = np.log1p(games["users_rated"])

listed_under = games["expansions"].explode().dropna().value_counts()
shared_items = set(listed_under[listed_under >= SHARED_ITEM_MIN_GAMES].index)
print(f"Expansion items: {len(listed_under):,}, {len(shared_items)} listed under {SHARED_ITEM_MIN_GAMES}+ base games")
print(listed_under.head(5).to_string())

# functions that checks if expansion is a core expansion
def is_core(item):
    return item not in shared_items and "promo" not in item.lower()

games["n_expansions_all"] = games["expansions"].map(len)
games["n_expansions"] = games["expansions"].map(lambda items: sum(is_core(i) for i in items))
games["has_expansion"] = (games["n_expansions"] > 0).astype(int)
games["has_expansion_all"] = (games["n_expansions_all"] > 0).astype(int)
games["log_n_expansions"] = np.log1p(games["n_expansions"])
games["expansion_bin"] = pd.cut(games["n_expansions"], bins=[-1, 0, 1, 3, 9, np.inf],
                                labels=["0", "1", "2-3", "4-9", "10+"])

games["expansion_bin"].value_counts(sort=False)

Expansion items: 21,803, 298 listed under 5+ base games
expansions
Super Power                        28
Brettspiel Adventskalender 2018    26
Brettspiel Adventskalender 2015    24
Brettspiel Adventskalender 2017    24
Brettspiel Adventskalender 2016    21


expansion_bin
0      15201
1       1999
2-3     1102
4-9      791
10+      452
Name: count, dtype: int64

**H3 moderator.** `planned_expansion_genre` = 1 if the game has one of the planned-expansion categories
or mechanics defined at the top. `wargame` is used to check whether H3 is driven by wargames alone.

In [120]:
def has_any(values):
    return lambda items: any(v in items for v in values)

planned_cat = games["categories"].map(has_any(PLANNED_CATEGORIES))
planned_mech = games["mechanics"].map(has_any(PLANNED_MECHANICS))

games["planned_expansion_genre"] = (planned_cat | planned_mech).astype(int)
games["wargame"] = games["categories"].map(has_any(["Wargame"])).astype(int)

print(f"Planned-expansion genre: {games['planned_expansion_genre'].mean():.1%} of games")

Planned-expansion genre: 22.6% of games


**Controls.** Playing time and maximum players are capped at the 99th percentile to account for impossibly high numbers (e.g. `maxplayers = 999`).
Publisher size = number of games of the first-listed publisher in the sample; self- and web-published
games get size 1 and a separate dummy. Game age is absorbed by publication-year fixed effects in the
models. `age` is only used for the descriptives.

In [121]:
for col in ["playingtime", "maxplayers"]:
    games[f"{col}_capped"] = games[col].clip(upper=games[col].quantile(0.99))
games["log_playingtime"] = np.log(games["playingtime_capped"])

publisher = games["publishers"].str[0]
games["self_published"] = (publisher.isna() | publisher.isin(NON_PUBLISHERS)).astype(int)
publisher_counts = publisher[games["self_published"] == 0].value_counts()
games["publisher_size"] = publisher.map(publisher_counts).fillna(1).astype(int)
games["log_publisher_size"] = np.log(games["publisher_size"])

games["age"] = SNAPSHOT_YEAR - games["yearpublished"]

**Exploratory dummies.** 0/1 dummies for the 15 most common categories and mechanics. The original
labels are saved in `dummy_labels.csv` for the figures in notebook 03.

In [122]:
labels = []
for column, prefix in [("categories", "cat"), ("mechanics", "mech")]:
    for value in games[column].explode().dropna().value_counts().head(TOP_N).index:
        name = prefix + "_" + re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")
        games[name] = games[column].map(lambda items: int(value in items))
        labels.append({"column": name, "family": column, "label": value})
labels = pd.DataFrame(labels)
labels

,column,family,label
0,cat_card_game,categories,Card Game
1,cat_wargame,categories,Wargame
2,cat_fantasy,categories,Fantasy
3,cat_party_game,categories,Party Game
4,cat_dice,categories,Dice
5,cat_children_s_game,categories,Children's Game
6,cat_science_fiction,categories,Science Fiction
7,cat_fighting,categories,Fighting
8,cat_abstract_strategy,categories,Abstract Strategy
9,cat_economic,categories,Economic


## 5. Sanity checks and save

Spot check: compare the expansion counts of well-known games with their BoardGameGeek pages.

In [123]:
known = ["Catan", "Dominion", "Ticket to Ride"]
games.loc[games["name"].isin(known), ["name", "yearpublished", "owned", "n_expansions_all", "n_expansions"]]

,name,yearpublished,owned,n_expansions_all,n_expansions
2,Catan,1995,167733,87,85
4,Dominion,2008,106956,46,33
5,Ticket to Ride,2004,105748,155,137


In [124]:
assert games["id"].is_unique
assert games["yearpublished"].between(MIN_YEAR, MAX_YEAR).all()
assert (games["owned"] >= 0).all()
assert (games["n_expansions"] <= games["n_expansions_all"]).all()

games.drop(columns=LIST_COLUMNS).to_csv(PROCESSED / "games_clean.csv", index=False)
labels.to_csv(PROCESSED / "dummy_labels.csv", index=False)
print(f"Saved {len(games):,} games to data/processed/games_clean.csv")

Saved 19,545 games to data/processed/games_clean.csv
